# Ask Alfonso · Phase 4 — training on a GPU

Trains the from-scratch transformer on Google Colab's free GPU.

**Before you start:** `Runtime → Change runtime type → T4 GPU → Save`.
Then run the cells in order (`Shift + Enter`). The whole notebook takes about 15–30 minutes.

Everything is rebuilt from the GitHub repo, so the model always trains on the latest `src/data`.

In [ ]:
# 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("PyTorch", torch.__version__, "· CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → T4 GPU"

In [ ]:
# 2. Get the code (re-running this cell pulls the latest version)
import os
REPO = "https://github.com/Alfonso3435/portfolio.git"
if not os.path.exists("/content/portfolio"):
    !git clone -q {REPO} /content/portfolio
%cd /content/portfolio
!git pull -q
!git log -1 --format="Commit %h · %s"

In [ ]:
# 3. Phase 1 — export the site data and build the dataset (needs Node.js for the TypeScript export)
!node --version || (curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null && apt-get install -y -qq nodejs)
!npm ci --ignore-scripts --no-audit --no-fund --loglevel=error
!npm run --silent export:knowledge
!python ml/build_dataset.py --samples 0

In [ ]:
# 4. Phase 2 — train the tokenizer (deterministic: identical to the one on your laptop)
!python ml/train_tokenizer.py | tail -8

### Optional: pretraining on general English (stage A)

Off by default. Turn it on to test whether learning general English first helps
— a good experiment for the Lab page (compare dev accuracy with and without it).
It downloads WikiText-2 (~11 MB of Wikipedia articles) from Hugging Face.

In [ ]:
# 5. (Optional) download pretraining text
PRETRAIN = False  # ← set to True to run stage A
PRETRAIN_ARGS = ""
if PRETRAIN:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], check=True)
    from datasets import load_dataset
    wiki = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
    with open("ml/data/wikitext2.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(t for t in wiki["text"] if t.strip()))
    PRETRAIN_ARGS = "--pretrain-text ml/data/wikitext2.txt --pretrain-steps 3000"
    print("Pretraining text ready.")

In [ ]:
# 6. Phase 4 — train (≈40–60 min on a T4). Every 4 epochs it prints "dev EM": the % of dev answers that are exactly right.
RUN = "colab-v2"
!python ml/train.py --run {RUN} {PRETRAIN_ARGS}

In [ ]:
# 7. Loss curves — saved as a PNG for the Lab page
import json
import matplotlib.pyplot as plt

rows = [json.loads(l) for l in open(f"ml/checkpoints/{RUN}/metrics.jsonl")]
ft = [r for r in rows if r["stage"] == "finetune"]
fig, (a, b) = plt.subplots(1, 2, figsize=(11, 4))
a.plot([r["epoch"] for r in ft], [r["train_loss"] for r in ft], label="train")
a.plot([r["epoch"] for r in ft], [r["dev_loss"] for r in ft], label="dev")
a.set(title="Loss (answer tokens)", xlabel="epoch", ylabel="cross-entropy"); a.legend()
em = [r for r in ft if "dev_em" in r]
b.plot([r["epoch"] for r in em], [100 * r["dev_em"] for r in em], color="tab:green", marker="o")
b.set(title="Dev exact match", xlabel="epoch", ylabel="% answers exactly right", ylim=(0, 100))
plt.tight_layout(); plt.savefig(f"ml/checkpoints/{RUN}/curves.png", dpi=150); plt.show()

In [ ]:
# 8. Ask it something. Retrieval comes in Phase 5, so for now you pick which chunk(s) it reads.
import sys; sys.path.insert(0, "ml")
from askalfonso.inference import answer, load_checkpoint
from askalfonso.training_data import build_context
chunks = {json.loads(l)["id"]: json.loads(l)["text"] for l in open("ml/data/chunks.jsonl")}
model, tok, meta = load_checkpoint(f"ml/checkpoints/{RUN}/best.pt")
print("Best:", meta["best"])

def ask(question, chunk_ids=()):
    print(f"Q: {question}\nA: {answer(model, tok, build_context(list(chunk_ids), chunks), question)}\n")

ask("What tech stack did he use for IdentifEye?", ["project/identifeye"])
ask("Did SkySAR win anything?", ["project/skysar", "award/thinkcity-2026"])
ask("Does he know Rust?", ["skills/languages"])
ask("What's the capital of France?")
print("Chunk ids you can use:", ", ".join(list(chunks)[:12]), "…")

In [ ]:
# 9. Download the trained model + metrics to your computer
!cd ml/checkpoints && zip -qr /content/{RUN}.zip {RUN}
from google.colab import files
files.download(f"/content/{RUN}.zip")